# PACKAGES

In [1]:
import pandas as pd
import numpy as np
import unicodedata
import re
import os
import gc
import io
import boto3
from pathlib import Path
from collections import defaultdict, Counter
from typing import Dict, Tuple, Optional, List
import warnings

warnings.filterwarnings('ignore')


# RENOMMAGE DES COLONNES

## Fonctions nécessaires

## Création des fonctions necessaires

In [2]:
# Je défini une fonction que je nomme "normalize_label"
def normalize_label(s: str) -> str:
    """Minuscules, accents retirés, ponctuation uniformisée, espaces compressés."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""  # uniquement si c'est un vrai NaN
    s = str(s).strip()
    s = (s.replace("-", " ")
           .replace("’", "'")
           .replace("'", " ")
           .replace(".", " "))
    s = s.lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"\s+", " ", s).strip()
    return s

# Je défini une fonction que je nomme "_pick_excel_engine" qui a pour objectif d’importer le module openpyxl.
# Si l’import réussit, la fonction renvoie "openpyxl"
# Si openpyxl n’est pas installé, Python va dans ce cas passer au cas exceptionnel. Essayer d’importer xlsxwriter.Si ça marche, il renvoie "xlsxwriter".

def _pick_excel_engine():
    try:
        import openpyxl  # noqa
        return "openpyxl"
    except Exception:
        try:
            import xlsxwriter  # noqa
            return "xlsxwriter"
        except Exception:
            raise RuntimeError("Installez 'openpyxl' ou 'xlsxwriter' pour écrire dans Excel.")

# Je défini une fonction que je nomme "write_sheet" qui a pour rôle d'écrire un DataFrame dans un fichier Excel,
# en remplaçant uniquement la feuille (sheet) spécifiée, sans effacer les autres feuilles du classeur

def write_sheet(df: pd.DataFrame, workbook_path: str, sheet_name: str):
    """
    Écrit/Remplace la feuille `sheet_name` dans `workbook_path`.
    - Si le classeur n'existe pas: création sans if_sheet_exists.
    - S'il existe: append + replace de la feuille.
    """
    engine = _pick_excel_engine()
    xlsx = Path(workbook_path)

    if engine == "openpyxl":
        if xlsx.exists():
            # append + replace la feuille
            with pd.ExcelWriter(workbook_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as xw:
                df.to_excel(xw, sheet_name=sheet_name, index=False)
        else:
            # création (pas de if_sheet_exists en mode 'w')
            with pd.ExcelWriter(workbook_path, engine="openpyxl", mode="w") as xw:
                df.to_excel(xw, sheet_name=sheet_name, index=False)
    else:
        # xlsxwriter: on doit recharger les feuilles existantes pour les réécrire
        sheets = {}
        if xlsx.exists():
            try:
                with pd.ExcelFile(workbook_path) as xf:
                    for sn in xf.sheet_names:
                        sheets[sn] = xf.parse(sn)
            except Exception:
                sheets = {}
        sheets[sheet_name] = df
        with pd.ExcelWriter(workbook_path, engine="xlsxwriter") as xw:
            for sn, d in sheets.items():
                d.to_excel(xw, sheet_name=sn, index=False)

    print(f"✅ Feuille écrite : {sheet_name} → {workbook_path}")



## Chargement + Paramètres + Vérifs de base

In [3]:
# Définition d'une fonction pour lire et nettoyer les fichiers Excel
def read_reference(
    excel_source,
    sheet_name: str,
    required_cols=None,          # ex: ["CODE_FONCTION","LIBELLÉ_FONCTION"]
    alias_map: dict | None = None, # alias optionnel {col_source: col_cible}, appliqué UNIQUEMENT ici
    drop_title_substr: str | None = "TABLEAU",  # supprime les lignes dont la 1ʳᵉ colonne contient ce motif
    preview_rows: int = 5,
) -> pd.DataFrame:
    """
    Lit une feuille Excel, affiche un aperçu AVANT/APRÈS nettoyage et retourne le DataFrame propre.

    Nettoyage :
      - supprime lignes entièrement vides
      - supprime lignes “titre” si la 1ʳᵉ colonne contient `drop_title_substr` (ex: 'TABLEAU')
      - si besoin, promeut la 1ʳᵉ ligne en entête
      - applique un alias de colonnes (optionnel, UNIQUEMENT au chargement)
      - vérifie la présence des colonnes `required_cols` si fourni
    """

    # --- Lecture brute ---
    df_raw = pd.read_excel(excel_source, sheet_name=sheet_name)

    print(f"\n=== {sheet_name} | APERÇU BRUT (top {preview_rows}) ===")
    try:
        from IPython.display import display
        display(df_raw.head(preview_rows))
    except Exception:
        print(df_raw.head(preview_rows).to_string(index=False))
    print(f"Colonnes BRUTES ({sheet_name}) :", df_raw.columns.tolist())
    print(f"Shape brut : {df_raw.shape}")

    # --- Copie de travail ---
    df = df_raw.copy()

    # 1) Supprimer les lignes entièrement vides
    df = df.dropna(how="all")

    # 2) Supprimer les lignes de titre si la 1ʳᵉ colonne les contient (ex: 'TABLEAU')
    if drop_title_substr is not None and not df.empty:
        first_col = df.columns[0]
        mask_title = df[first_col].astype(str).str.contains(drop_title_substr, case=False, na=False)
        df = df.loc[~mask_title].copy()

    # 3) Re-détecter l'entête si nécessaire (heuristique simple)
    def _looks_like_header(row_vals, required, alias):
        vals = [str(v).strip() for v in row_vals]
        if required and any(c in vals for c in required):
            return True
        if alias and any(src in vals for src in alias.keys()):
            return True
        if any("unnamed" in v.lower() for v in vals):
            return False
        if any(("code" in v.lower()) or ("lib" in v.lower()) for v in vals):
            return True
        return False

    if not df.empty:
        candidate = df.iloc[0].tolist()
        if _looks_like_header(candidate, required_cols, alias_map):
            df.columns = [str(v).strip() for v in candidate]
            df = df.drop(df.index[0]).reset_index(drop=True)

    # 4) Alias de colonnes (optionnel, seulement au chargement)
    if alias_map:
        df = df.rename(columns=alias_map)

    # 5) Vérifier les colonnes requises (si fourni)
    if required_cols:
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            print("\n⚠️ Colonnes manquantes après nettoyage/alias :", missing)
            print("Colonnes disponibles :", df.columns.tolist())
            raise ValueError(f"Colonnes manquantes : {missing}")

    # 6) Aperçu après nettoyage
    print(f"\n=== {sheet_name} | APRÈS NETTOYAGE (top {preview_rows}) ===")
    try:
        from IPython.display import display
        display(df.head(preview_rows))
    except Exception:
        print(df.head(preview_rows).to_string(index=False))
    print(f"Colonnes NETTOYÉES ({sheet_name}) :", df.columns.tolist())
    print(f"Shape nettoyé : {df.shape}")

    return df.reset_index(drop=True)


def check_code_to_label_uniqueness(df_ref: pd.DataFrame, var_code_ext: str, var_lib_ext: str) -> pd.Series:
    """
    Vérifie que chaque code (var_code_ext) pointe vers un seul libellé (var_lib_ext).
    Retourne la série des codes non-uniques (counts).
    """
    verif_code = df_ref.groupby(var_code_ext)[var_lib_ext].nunique()
    bad = verif_code[verif_code > 1]
    if bad.empty:
        print("✅ Chaque code est associé à un seul libellé.")
    else:
        print("❌ Codes associés à plusieurs libellés :")
        print(bad)
    return bad

def ensure_normalized_column(df_ref: pd.DataFrame, var_lib_ext: str, var_norm_ext: str | None = None) -> str:
    """
    Ajoute la colonne normalisée (var_norm_ext) si absente. Retourne son nom.
    """
    var_norm_ext = var_norm_ext or f"{var_lib_ext}_NORM"
    if var_norm_ext not in df_ref.columns:
        df_ref[var_norm_ext] = df_ref[var_lib_ext].apply(normalize_label)
    return var_norm_ext

def detect_normalized_duplicates(df_ref: pd.DataFrame,
                                 var_norm_ext: str,
                                 var_lib_ext: str,
                                 var_code_ext: str) -> pd.DataFrame:
    """
    Détecte les doublons après normalisation et retourne toutes les lignes uniques concernées.

    Paramètres
    ----------
    df_ref : pd.DataFrame
        Table de référence contenant les codes et libellés normalisés.
    var_norm_ext : str
        Nom de la colonne contenant les libellés normalisés.
    var_lib_ext : str
        Nom de la colonne contenant les libellés originaux.
    var_code_ext : str
        Nom de la colonne contenant les codes associés.

    Retour
    ------
    pd.DataFrame
        Toutes les lignes uniques dont la clé normalisée apparaît plusieurs fois.
    """
    # Compter combien de fois chaque clé normalisée apparaît
    counts = df_ref[var_norm_ext].value_counts()
    repeated_keys = counts[counts > 1].index.tolist()

    if not repeated_keys:
        print("✅ Aucun doublon détecté après normalisation.")
        return pd.DataFrame(columns=[var_code_ext, var_lib_ext, var_norm_ext])

    # Filtrer toutes les lignes concernées
    dup_rows = df_ref[df_ref[var_norm_ext].isin(repeated_keys)].copy()

    # Garder uniquement les lignes uniques (code + libellé + normalisé)
    dup_rows = dup_rows.drop_duplicates(subset=[var_code_ext, var_lib_ext, var_norm_ext])

    # Message d’alerte clair
    print(f"⚠️ Doublons détectés après normalisation : {len(repeated_keys)} clés répétées, {len(dup_rows)} lignes uniques concernées.")

    return dup_rows

In [4]:
def report_duplicates(df: pd.DataFrame, var_norm: str, var_lib: str, var_code: str) -> pd.DataFrame:
    """
    Affiche la liste exhaustive des doublons après normalisation et retourne toutes les lignes concernées.
    
    Paramètres
    ----------
    df : pd.DataFrame
        Table contenant les libellés et codes.
    var_norm : str
        Colonne avec les libellés normalisés (clé de regroupement).
    var_lib : str
        Colonne avec les libellés originaux.
    var_code : str
        Colonne avec les codes associés.

    Retour
    ------
    pd.DataFrame
        Toutes les lignes du DataFrame où une même clé normalisée apparaît plusieurs fois.
    """

    # --- 1) Compter toutes les occurrences de la clé normalisée ---
    counts = df[var_norm].value_counts()

    # --- 2) Sélectionner les clés répétées ---
    repeated_keys = counts[counts > 1].index.tolist()

    if not repeated_keys:
        print("✅ Aucun doublon détecté après normalisation.")
        return pd.DataFrame(columns=[var_code, var_lib, var_norm])

    # --- 3) Filtrer toutes les lignes concernées ---
    dup_rows = df[df[var_norm].isin(repeated_keys)].copy()

    # --- 4) Message d’alerte ---
    print(f"⚠️ Doublons détectés après normalisation : {len(repeated_keys)} clés répétées, {len(dup_rows)} lignes concernées.")

    # --- 5) Affichage des colonnes essentielles pour l’audit ---
    try:
        from IPython.display import display
        display(dup_rows[[var_code, var_lib, var_norm]])
    except Exception:
        print(dup_rows[[var_code, var_lib, var_norm]].to_string(index=False))

    # --- 6) Retourner le DataFrame complet ---
    return dup_rows

## Construction du mapping (1 code par libellé normalisé)

In [5]:
# La fonction "build_random_mapping" a pour but de créer une association unique entre des codes et les libellés normalisés dans un DataFrame,
# en gérant les ambiguïtés et en ajoutant d'autres informations

def build_random_mapping(
    df_ref: pd.DataFrame, # Le DataFrame source contenant les données (codes, libellés brutww, libellés normalisés)
    var_norm_ext: str, # Nom de la colonne "libellé normalisé" sur laquelle on veut créer le mapping
    var_code_ext: str, # Nom de la colonne "code" qui sera associé au libellé normalisé
    var_lib_ext: str, # Nom de la colonne "libellé brut" correspondant au libellé de départ avant traitement
    seed: int = 123 # Seed pour rendre le tirage aléatoire reproductible
) -> pd.DataFrame:
    """
    Construit un mapping 1 code / clé normalisée :
      - choisit aléatoirement un code quand plusieurs existent (traçable par seed),
      - ajoute les colonnes d’audit : EST_AMBIGU, NB_CODES_POSSIBLES, CODES_POSSIBLES.
    Sortie : DataFrame avec colonnes [var_lib_ext, var_norm_ext, var_code_ext,
                                     EST_AMBIGU, NB_CODES_POSSIBLES, CODES_POSSIBLES, CODE_CHOISI]
    """
    df = df_ref.copy() # On crée une copie du DataFrame d’origine
    # infos par clé
    codes_list = (df.groupby(var_norm_ext)[var_code_ext]
                    .apply(lambda s: Asorted(map(str, pd.unique(s))))
                    .rename("CODES_LIST")) # créer pour chaque libellé normalisé (var_norm_ext) la liste des codes uniques associés (var_code_ext).
    codes_info = codes_list.to_frame()
    codes_info["NB_CODES_POSSIBLES"] = codes_info["CODES_LIST"].str.len() # Compte le nombre de codes possibles pour chaque libellé normalisé
    codes_info["EST_AMBIGU"] = codes_info["NB_CODES_POSSIBLES"].gt(1) #Crée une colonne booléenne indiquant si le libellé normalisé est associé à plusieurs codes.1" → True si plus d’un code, False sinon
    codes_info["CODES_POSSIBLES"] = codes_info["CODES_LIST"].apply(lambda L: ", ".join(L))

    # tirage aléatoire d'un code par clé normalisée
    pairs_uniques = df[[var_norm_ext, var_code_ext]].drop_duplicates() # On prend seulement les colonnes libellé normalisé (var_norm_ext) et code (var_code_ext) du DataFrame. Ensuite, on supprime les lignes identiques pour ne garder qu’une seule occurrence par combinaison
    selection = (pairs_uniques.groupby(var_norm_ext, group_keys=False)
                 .apply(lambda g: g.sample(n=1, random_state=seed))
                 .reset_index(drop=True)
                 .rename(columns={var_code_ext: "CODE_CHOISI"}))
    # groupby(var_norm_ext, group_keys=False) : On regroupe les lignes par libellé normalisé et chaque groupe contient tous les codes possibles pour ce libellé.
    # .apply(lambda g: g.sample(n=1, random_state=seed)) : pour chaque libellé, on choisit aléatoirement un code parmi les codes possibles. random_state=seed permet que le tirage soit répétable : même code choisi si on réexécute le programme avec le même seed. reset_index(drop=True) : Réinitialise l’index du DataFrame résultant pour avoir un index simple allant de 0 à n-1.
    # .rename(columns={var_code_ext: "CODE_CHOISI"}) : renomme la colonne du code choisi pour clarifier que c’est celui sélectionné aléatoirement.

    # mapping final = libellé brut x clé norm + code choisi + audit d’ambiguïté
    mapping = (df[[var_lib_ext, var_norm_ext]]
               .drop_duplicates() 
               # On récupère les colonnes libellé brut (var_lib_ext) et libellé normalisé (var_norm_ext). Ensuite, on supprime les doublons pour avoir une seule ligne par combinaison libellé brut / libellé normalisé.
               
               .merge(selection, on=var_norm_ext, how="left") 
               # On ajoute la colonne CODE_CHOISI à chaque libellé normalisé. un code choisi aléatoirement par libellé normalisé.
               
               .merge(codes_info.drop(columns=["CODES_LIST"]), left_on=var_norm_ext, right_index=True, how="left"))
                # On ajoute les informations d’audit (nombre de codes possibles, liste des codes, indicateur d’ambiguïté) depuis codes_info. on supprime la liste brute des codes pour ne garder que les colonnes utiles pour l’audit.
    mapping[var_code_ext] = mapping["CODE_CHOISI"]

    cols = [var_lib_ext, var_norm_ext, var_code_ext, "CODE_CHOISI", "EST_AMBIGU",
            "NB_CODES_POSSIBLES", "CODES_POSSIBLES"]
    mapping = mapping[cols].sort_values([ "EST_AMBIGU", var_norm_ext, var_lib_ext ],
                                        ascending=[False, True, True])
    return mapping

def export_mapping(mapping: pd.DataFrame, workbook_path: str, sheet_name: str):
    """
    Écrit UNIQUEMENT la feuille de mapping (pas de feuille ‘Ambigus’) ;
    la feuille contient les colonnes d’audit ‘EST_AMBIGU’, ‘CODES_POSSIBLES’, etc.
    """
    write_sheet(mapping, workbook_path, sheet_name)


## Fusion du mapping dans le panel + bilan des non appariés

In [6]:
def merge_mapping_into_panel(panel_df: pd.DataFrame,
                             mapping: pd.DataFrame,
                             var_norm_panel: str, var_norm_ext: str,
                             var_code_panel: str, var_code_ext: str,
                             var_lib_panel: str):
    """
    Met à jour les codes du panel à partir du mapping via les libellés normalisés.
    Le code est récupéré directement du mapping et prend le nom de la colonne du panel.
    """
    # 1️⃣ Copie du panel
    df = panel_df.copy()

    # 2️⃣ Créer la colonne normalisée dans le panel si absente
    if var_norm_panel not in df.columns:
        df[var_norm_panel] = df[var_lib_panel].apply(normalize_label)

    # 3️⃣ Préparer le mapping (suppression doublons)
    mapping_clean = mapping[[var_norm_ext, var_code_ext]].drop_duplicates()
    mapping_clean[var_code_ext] = mapping_clean[var_code_ext].astype("string").fillna("NA")

    # 4️⃣ Merge sur la clé normalisée et renommer la colonne du code pour matcher le panel
    out = df.merge(mapping_clean,
                   left_on=var_norm_panel,
                   right_on=var_norm_ext,
                   how="left")\
            .rename(columns={var_code_ext: var_code_panel})

    # 5️⃣ Supprimer les colonnes techniques issues du merge
    out = out.drop(columns=[var_norm_ext], errors="ignore")

    # 6️⃣ Statistiques
    nb_rens = int(out[var_code_panel].notna().sum())
    nb_miss = int(out[var_code_panel].isna().sum())
    print(f"Codes renseignés dans {var_code_panel} : {nb_rens}")
    print(f"Codes manquants dans {var_code_panel} : {nb_miss}")

    return out, nb_rens, nb_miss


## Similarité sur les non appariés (top 1 par clé)

In [7]:
#  la fonction "list_unmatched_norms" liste toutes les libellés normalisés (var_norm_panel) dans le panel qui n’ont pas de code associé (var_code_panel vide ou NA).
# Cela permet de détecter les éléments du panel qui n’ont pas pu être appariés avec le mapping.


''' Définition d’une fonction qui prend : 
(1) panel_df  qui est le tableau de données principal (panel)
(2) var_code_panel : la colonne du code qui peut être manquante (NA)
(3) var_norm_panel : la colonne du libellé normalisé '''

def list_unmatched_norms(panel_df: pd.DataFrame, var_code_panel: str, var_norm_panel: str) -> pd.DataFrame:
    """
    Retourne les libellés normalisés non appariés avec le nombre de lignes correspondantes,
    y compris les libellés vides ou entièrement blancs.
    """
    # Sélection des lignes où le code est NA
    s = panel_df.loc[panel_df[var_code_panel].isna(), var_norm_panel].astype(str)

    # Nettoyer les chaînes (supprimer espaces en début/fin)
    s = s.str.strip()

    # Compter les occurrences de chaque libellé, y compris les vides
    counts = s.value_counts(dropna=False).reset_index()
    counts.columns = [var_norm_panel, "COUNT"]

    if counts.empty:
        print("✅ Tous les libellés ont été appariés, aucun non apparié.")
    else:
        total_non_apparies = counts["COUNT"].sum()
        print(f"⚠️ {len(counts)} libellés non appariés trouvés (total lignes concernées : {total_non_apparies}) :")
        try:
            from IPython.display import display
            display(counts)
        except Exception:
            print(counts.to_string(index=False))

    return counts
        
# Cette fonction "best_suggestions" cherche pour chaque libellé non apparié le meilleur candidat dans le référentiel
def best_suggestions(panel_df: pd.DataFrame, mapping: pd.DataFrame,
                     var_norm_panel: str, var_norm_ext: str,
                     var_lib_ext: str, var_code_ext: str,
                     var_code_panel: str,
                     top_n: int = 20,
                     min_score_show: int = 0) -> pd.DataFrame:
    """
    Pour chaque libellé normalisé non apparié dans le panel, calcule le MEILLEUR candidat
    du référentiel (score de similarité). Affiche le top et retourne un DataFrame suggestions_best.

    Paramètres
    ----------
    panel_df : pd.DataFrame
        DataFrame du panel principal (contenant les codes éventuellement manquants)
    mapping : pd.DataFrame
        Référentiel avec les codes et libellés normalisés
    var_norm_panel : str
        Nom de la colonne des libellés normalisés dans le panel
    var_norm_ext : str
        Nom de la colonne des libellés normalisés dans le référentiel
    var_lib_ext : str
        Nom de la colonne des libellés originaux dans le référentiel
    var_code_ext : str
        Nom de la colonne des codes dans le référentiel
    var_code_panel : str
        Nom de la colonne des codes dans le panel
    top_n : int
        Nombre maximum de suggestions à afficher
    min_score_show : int
        Seuil minimal pour afficher les suggestions (score de similarité)
    
    Retour
    ------
    pd.DataFrame
        DataFrame avec les meilleures suggestions pour chaque libellé non apparié.
        Colonnes : [f"{var_norm_panel}_NON_APPARIE", f"MATCH_{var_norm_ext}", "MATCH_SCORE",
                    var_lib_ext, var_code_ext]
    """

    # --- 1) Extraire les libellés non appariés dans le panel ---
    # Utilise list_unmatched_norms qui retourne COUNT, mais ici on prend uniquement la liste
    non_apparies_df = list_unmatched_norms(panel_df, var_code_panel, var_norm_panel)
    non_apparies = non_apparies_df[var_norm_panel].tolist()  # conversion en liste simple

    # --- 2) Préparer la liste des clés normalisées du référentiel ---
    ref_norms = mapping[var_norm_ext].dropna().astype(str).str.strip()
    ref_norms = ref_norms[ref_norms.ne("")].unique().tolist()  # enlever les vides et doublons

    # --- 3) Déterminer le moteur de similarité ---
    use_rapidfuzz = False
    try:
        from rapidfuzz import process, fuzz  # type: ignore
        use_rapidfuzz = True  # RapidFuzz disponible pour calculer la similarité rapidement
    except Exception:
        from difflib import SequenceMatcher  # fallback si RapidFuzz non installé

    # --- 4) Fonction interne pour calculer le meilleur match d’un libellé ---
    def _best_match(q: str):
        if not q or q.strip() == "":
            return None  # pas de texte → pas de match possible
        if use_rapidfuzz:
            # renvoie (libellé normalisé correspondant, score, index)
            return process.extractOne(q, ref_norms, scorer=fuzz.token_set_ratio)
        else:
            # fallback : utiliser SequenceMatcher pour comparer chaque libellé
            best_score, best_idx = -1, None
            for idx, rn in enumerate(ref_norms):
                sc = int(SequenceMatcher(None, q, rn).ratio() * 100)
                if sc > best_score:
                    best_score, best_idx = sc, idx
            return (ref_norms[best_idx], best_score, best_idx) if best_idx is not None else None

    # --- 5) Chercher le meilleur candidat pour chaque libellé non apparié ---
    rows = []
    for qn in non_apparies:
        res = _best_match(qn)

        if res is None:
            # Cas où le libellé est vide ou aucun match trouvé
            rows.append({
                f"{var_norm_panel}_NON_APPARIE": qn,
                f"MATCH_{var_norm_ext}": "AUCUNE_SUGGESTION",
                "MATCH_SCORE": 0,
                var_lib_ext: "Aucune suggestion possible",
                var_code_ext: None
            })
            continue

        matched_norm, score, _ = res

        # récupérer la ligne correspondante dans le mapping (libellé et code)
        # si plusieurs correspondances, on prend la première pour la stabilité
        cand = mapping.loc[mapping[var_norm_ext] == matched_norm, [var_lib_ext, var_code_ext]].head(1)
        if cand.empty:
            rows.append({
                f"{var_norm_panel}_NON_APPARIE": qn,
                f"MATCH_{var_norm_ext}": matched_norm,
                "MATCH_SCORE": int(score),
                var_lib_ext: "Non trouvé dans mapping",
                var_code_ext: None
            })
            continue

        lib_val = cand.iloc[0][var_lib_ext]
        code_val = cand.iloc[0][var_code_ext]

        # construire une ligne pour le DataFrame des suggestions
        rows.append({
            f"{var_norm_panel}_NON_APPARIE": qn,
            f"MATCH_{var_norm_ext}": matched_norm,
            "MATCH_SCORE": int(score),
            var_lib_ext: lib_val,
            var_code_ext: code_val,
        })

    # --- 6) Colonnes attendues ---
    expected_cols = [
        f"{var_norm_panel}_NON_APPARIE",
        f"MATCH_{var_norm_ext}",
        "MATCH_SCORE",
        var_lib_ext,
        var_code_ext,
    ]

    # --- 7) Création du DataFrame final ---
    suggestions_best = pd.DataFrame(rows)

    # Trier par score puis par libellé pour afficher les meilleures suggestions en haut
    if not suggestions_best.empty:
        suggestions_best = suggestions_best.sort_values(
            ["MATCH_SCORE", var_lib_ext], ascending=[False, True]
        )[expected_cols]
        if min_score_show > 0:
            to_show = suggestions_best[suggestions_best["MATCH_SCORE"] >= min_score_show].head(top_n)
        else:
            to_show = suggestions_best.head(top_n)
    else:
        # DataFrame vide structuré avec les colonnes attendues
        suggestions_best = pd.DataFrame(columns=expected_cols)
        to_show = suggestions_best

    # --- 8) Affichage Jupyter-friendly ---
    try:
        from IPython.display import display
        display(to_show)
    except Exception:
        print(to_show.to_string(index=False))

    return suggestions_best


## Application des remplacements (manuel OU automatique)

In [8]:
def apply_choices(panel_df: pd.DataFrame,
                  var_code_panel: str, var_norm_panel: str,
                  choices_map: dict[str, str],
                  var_cle_panel: str, var_lib_panel: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Applique un dictionnaire {clé_norm_panel -> code}, uniquement là où le code est NA.
    Renvoie (panel_df_mis_a_jour, audit_df).
    """
    df = panel_df.copy()
    # Masque des lignes où le code est manquant
    fill_mask = df[var_code_panel].isna()

    # Colonne temporaire pour stocker les suggestions de code
    sugg_col = f"{var_code_panel}_SUGG"
    df.loc[fill_mask, sugg_col] = df.loc[fill_mask, var_norm_panel].map(choices_map)

    # Masque des lignes où : var_code_panel est encore NA mais une suggestion a été trouvée dans sugg_col
    m_apply = df[var_code_panel].isna() & df[sugg_col].notna()
    
    # Création d’une colonne BEFORE pour garder une trace de l’état initial des codes avant mise à jour.
    before_col = f"{var_code_panel}_BEFORE" 
    df[before_col] = df[var_code_panel]
    

    # Remplit les codes manquants avec les valeurs suggérées.
    df.loc[m_apply, var_code_panel] = df.loc[m_apply, sugg_col] 
    
    '''
    Crée un DataFrame d’audit qui contient :

           - la clé et le libellé d’origine
           - le code avant mise à jour
           - le code après mise à jour (AFTER)

     Cela permet de suivre exactement quelles lignes ont été modifiées.
     '''

    audit = (df.loc[m_apply, [var_cle_panel, var_lib_panel, var_norm_panel, before_col, var_code_panel]]
               .copy()
               .rename(columns={var_code_panel: f"{var_code_panel}_AFTER"}))

    print(f"✔ Lignes mises à jour : {len(audit)}")
    df.drop(columns=[sugg_col], errors="ignore", inplace=True)
    return df, audit



In [9]:
"""
Automatiser l’application des suggestions de codes sur le panel uniquement pour les suggestions avec un score de similarité supérieur à un seuil min_score.
Elle utilise la fonction apply_choices pour remplir les codes manquants.
Définition de la fonction avec les paramètres :

     - panel_df : le tableau de données principal où on veut remplir les codes.
     - suggestions_best : le DataFrame contenant les suggestions de codes avec score de similarité.
     - var_code_panel : colonne du code dans le panel (celle à remplir).
     - var_norm_panel : colonne du libellé normalisé dans le panel.
     - var_cle_panel : colonne contenant une clé d’identification unique pour chaque ligne.
     - var_lib_panel : colonne du libellé original dans le panel.
     - var_code_ext : colonne du code dans le référentiel de suggestions.
     - min_score : seuil minimal pour appliquer automatiquement la suggestion (défaut 70).

La fonction retourne un tuple (panel_df_mis_a_jour, audit_df).
"""
def auto_apply_by_score(panel_df: pd.DataFrame,
                        suggestions_best: pd.DataFrame,
                        var_code_panel: str, var_norm_panel: str,
                        var_cle_panel: str, var_lib_panel: str,
                        var_code_ext: str,
                        min_score: int = 70) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Construit choices_map pour toutes les suggestions avec score ≥ min_score, et applique.
    """
    if suggestions_best.empty: # Vérifie si le DataFrame des suggestions est vide.
        print("INFO : pas de suggestions à appliquer.")
        return panel_df.copy(), pd.DataFrame()  # Si oui : on affiche un message, et on retourne une copie du panel inchangé et un DataFrame d’audit vide.

    # nom de la colonne 'libelle_norm_non_apparie' dans suggestions_best
    norm_non_app_col = [c for c in suggestions_best.columns if c.endswith("_NON_APPARIE")]
    if not norm_non_app_col:
        raise ValueError("La table suggestions_best ne contient pas de colonne *_NON_APPARIE.")
    norm_non_app_col = norm_non_app_col[0]

    # Filtre les suggestions pour ne garder que celles dont le score de similarité est supérieur ou égal à min_score. Seules ces suggestions seront appliquées
    ok = suggestions_best[suggestions_best["MATCH_SCORE"] >= min_score]

    # Construit un dictionnaire clé → code à partir des suggestions filtrées : la clé est le libellé normalisé non apparié dans le panel (norm_non_app_col). la valeur est le code correspondant dans le référentiel (var_code_ext).
    # Ce dictionnaire sera utilisé par apply_choices pour remplir automatiquement les codes.
    choices_map = dict(zip(ok[norm_non_app_col], ok[var_code_ext]))

    return apply_choices(panel_df, var_code_panel, var_norm_panel, choices_map, var_cle_panel, var_lib_panel)


In [10]:
def enrich_mapping_with_choices(mapping_final: pd.DataFrame,
                                suggestions_best: pd.DataFrame,
                                var_norm_panel: str,
                                var_norm_ext: str, var_code_ext: str, var_lib_ext: str,
                                choices_map: dict[str, str]) -> pd.DataFrame:
    """
    Ajoute dans le mapping les alias normalisés retenus (clé_norm_panel -> code),
    en remplaçant l’éventuelle ancienne ligne pour cette clé.
    """
    if suggestions_best.empty or not choices_map:
        print("INFO : rien à enrichir (pas de suggestions ou pas de choix).")
        return mapping_final

    norm_non_app_col = [c for c in suggestions_best.columns if c.endswith("_NON_APPARIE")]
    if not norm_non_app_col:
        raise ValueError("suggestions_best ne contient pas de colonne *_NON_APPARIE.")
    norm_non_app_col = norm_non_app_col[0]

    # ne garder que les suggestions dont la clé est choisie
    sel = suggestions_best[suggestions_best[norm_non_app_col].isin(list(choices_map.keys()))].copy()

    # remplacer le code par celui imposé par choices_map (au cas où)
    sel[var_code_ext] = sel[norm_non_app_col].map(choices_map)

    add = (sel[[norm_non_app_col, var_code_ext, var_lib_ext]]
             .drop_duplicates()
             .rename(columns={norm_non_app_col: var_norm_ext}))

    need_cols = {var_norm_ext, var_lib_ext, var_code_ext}
    missing = [c for c in need_cols if c not in mapping_final.columns]
    if missing:
        raise ValueError(f"mapping_final doit contenir {need_cols}, manquant : {missing}")

    before = len(mapping_final)
    # retire les anciennes lignes pour ces clés
    mapping_final = mapping_final[~mapping_final[var_norm_ext].isin(add[var_norm_ext])]
    # concatène + dédoublonne
    mapping_final = (pd.concat([mapping_final, add], ignore_index=True)
                     .drop_duplicates(subset=[var_norm_ext, var_code_ext]))
    print(f"🔁 Mapping enrichi : +{len(mapping_final)-before} lignes (net).")
    return mapping_final

## Fonction uniquement pour la composition du salaire

In [11]:
# Cette fonction prend : df : un DataFrame Pandas et cols : une liste de colonnes à vérifier.
# Elle renvoie un DataFrame booléen de la même forme, indiquant si chaque cellule est “renseignée” (c’est-à-dire non vide).

def _filled_mask_for_cols(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    """
    Renvoie un DataFrame booléen (mêmes lignes, colonnes=cols) indiquant si chaque cellule est 'renseignée'.
    - Pour colonnes texte: non-NaN ET trim != ""
    - Pour colonnes non-texte: non-NaN
    """
    
    # On crée un sous-DataFrame sub avec uniquement les colonnes à vérifier.
    # --- Sélection des colonnes à vérifier ---
    sub = df[cols]
    filled = sub.notna()  # Tout sauf NaN est considéré comme renseigné. True si la cellule n’est pas NaN, False sinon.

    # Colonnes texte (object, string dtype)
    text_cols = [c for c in cols if c in sub.columns and is_string_dtype(sub[c])]
    if text_cols:
        # Pour ces colonnes: trim et vérifie non vide
        # Pas de .str sur DataFrame -> on applique par colonne (vectorisé)
        trimmed_ne_empty = sub[text_cols].apply(lambda s: s.str.strip().ne(""), axis=0)
        filled[text_cols] = sub[text_cols].notna() & trimmed_ne_empty

    return filled

In [12]:
"""
Cette fonction :

1. Vérifie que toutes les colonnes à analyser existent dans le DataFrame.
2. Utilise _filled_mask_for_cols pour savoir si chaque cellule est renseignée.
3. Compte pour chaque ligne combien de colonnes sont renseignées.
4. Affiche un résumé détaillé du nombre de colonnes renseignées par ligne :
   - Nombre total de lignes
   - Lignes correctes (exactement 1 colonne renseignée)
   - Lignes à problème (0 ou plusieurs colonnes renseignées)
   - Index des lignes problématiques
5. Retourne la liste des lignes qui n’ont pas exactement une seule colonne renseignée,
   ce qui peut poser problème pour les règles de complétude de données.
"""
def check_completeness(df: pd.DataFrame, cols: list[str], title: str) -> list[int]:
    # 1️⃣ Vérification de l’existence des colonnes
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"Colonnes manquantes pour [{title}]: {missing}")

    # 2️⃣ Masque booléen des cellules renseignées
    filled = _filled_mask_for_cols(df, cols)
    
    # 3️⃣ Compte combien de colonnes sont renseignées par ligne
    nb_renseignees = filled.sum(axis=1)

    # 4️⃣ Sélection des lignes problématiques (pas exactement 1 colonne renseignée)
    problemes_idx = df.index[nb_renseignees != 1].tolist()

    # 5️⃣ Résumé détaillé
    nb_total = len(df)
    nb_ok = nb_total - len(problemes_idx)
    print(f"\n📊 Résumé de la vérification [{title}] :")
    print(f"🔹 Nombre total de lignes : {nb_total}")
    print(f"✅ Lignes correctes (exactement 1 colonne renseignée) : {nb_ok}")
    print(f"⚠️ Lignes à problème (0 ou plusieurs colonnes renseignées) : {len(problemes_idx)}")
    print(f"📌 Index des lignes problématiques : {problemes_idx}")

    # 6️⃣ Retourne les index des lignes à problème
    return problemes_idx

In [13]:
def drop_if_other_cols_empty(
    df: pd.DataFrame,
    index_to_check: list[int],
    keep_cols: list[str],
    title: str = ""
) -> pd.DataFrame:
    """
    Supprime les lignes dont l'index est dans `index_to_check`
    ET où toutes les colonnes autres que `keep_cols` sont vides (NaN ou "" pour texte).
    """
    # Évite les doublons d'index et garde uniquement ceux qui existent
    idx_check = pd.Index(index_to_check).intersection(df.index)

    # Colonnes à vérifier (toutes sauf keep_cols)
    check_cols = [c for c in df.columns if c not in set(keep_cols)]
    if not check_cols or idx_check.empty:
        print(f"\n🗑️ [{title}] Rien à supprimer (colonnes à vérifier vides ou aucun index valide).")
        return df.reset_index(drop=True)

    # On calcule le masque 'vide' par colonne, SANS convertir tout en string
    sub = df.loc[idx_check, check_cols]

    # Colonnes texte
    text_cols = [c for c in check_cols if is_string_dtype(df[c])]
    empty_text = pd.DataFrame(index=idx_check)
    if text_cols:
        # vide si NaN OU trim == ""
        txt = sub[text_cols]
        empty_text = txt.isna() | txt.apply(lambda s: s.str.strip().eq(""), axis=0)
    else:
        # DataFrame vide mais avec bon index
        empty_text = pd.DataFrame(index=idx_check)

    # Colonnes non-texte: vide si NaN
    non_text_cols = [c for c in check_cols if c not in text_cols]
    empty_nontext = sub[non_text_cols].isna() if non_text_cols else pd.DataFrame(index=idx_check)

    # Concat puis 'all' sur les colonnes pour savoir si toute la ligne est vide côté "autres colonnes"
    if not empty_text.empty and not empty_nontext.empty:
        mask_all_other_empty = pd.concat([empty_text, empty_nontext], axis=1).all(axis=1)
    elif not empty_text.empty:
        mask_all_other_empty = empty_text.all(axis=1)
    elif not empty_nontext.empty:
        mask_all_other_empty = empty_nontext.all(axis=1)
    else:
        # Aucun check_cols (cas déjà géré plus haut, par sécurité)
        mask_all_other_empty = pd.Series(False, index=idx_check)

    final_drop_idx = mask_all_other_empty[mask_all_other_empty].index

    n_all = len(df)
    n_drop = len(final_drop_idx)
    print(f"\n🗑️ [{title}] Suppression de {n_drop} / {n_all} lignes (après vérif des autres colonnes vides)")

    # IMPORTANT: ne copie pas tout le DF inutilement
    return df.drop(index=final_drop_idx).reset_index(drop=True)

In [14]:
import pandas as pd
from pandas.api.types import is_string_dtype

def drop_if_other_cols_empty_ultralight(
    df: pd.DataFrame,
    index_to_check: list[int],
    keep_cols: list[str],
    title: str = "",
    reset_index: bool = False,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Version ultra légère pour grands DataFrames.
    """

    idx_check = pd.Index(index_to_check).intersection(df.index)
    if idx_check.empty:
        if verbose: print(f"\n🗑️ [{title}] Aucun index à vérifier.")
        return df if not reset_index else df.reset_index(drop=True)

    check_cols = [c for c in df.columns if c not in set(keep_cols)]
    if not check_cols:
        if verbose: print(f"\n🗑️ [{title}] Aucune colonne à vérifier.")
        return df if not reset_index else df.reset_index(drop=True)

    sub = df.loc[idx_check, check_cols]

    # Colonnes texte : vide si NaN ou après strip == ""
    mask_text = pd.Series(True, index=sub.index)
    for col in sub.columns:
        if is_string_dtype(df[col]):
            mask_text &= sub[col].isna() | (sub[col].str.strip() == "")

    # Colonnes non-texte : vide si NaN
    mask_nontext = sub.drop(columns=[c for c in sub.columns if is_string_dtype(df[c])], errors="ignore").isna().all(axis=1)

    # Masque final
    mask_all_other_empty = mask_text & mask_nontext
    final_drop_idx = mask_all_other_empty[mask_all_other_empty].index

    df_clean = df.drop(index=final_drop_idx)

    if verbose:
        print(f"\n🗑️ [{title}] Suppression de {len(final_drop_idx)} / {len(df)} lignes.")

    return df_clean.reset_index(drop=True) if reset_index else df_clean


In [15]:
def create_montant_et_matricule(
    df: pd.DataFrame,
    montant_cols: list[str],
    matricule_cols: list[str],
    montant_out: str = "MONTANT",
    matricule_out: str = "MATRICULE_UNIQUE",
    in_place: bool = True,
) -> pd.DataFrame:
    """
    Crée deux colonnes :
      - `montant_out` : première valeur non vide parmi `montant_cols`
                        puis convertie en numérique (NaN si non convertible)
      - `matricule_out`: première valeur non vide parmi `matricule_cols` (après trim)

    Priorité = ordre des listes.
    `in_place=True` modifie le DataFrame d'entrée, sinon retourne une copie.
    """
    def _dedupe(seq):  # conserve l'ordre, supprime doublons
        return list(dict.fromkeys(seq))

    montant_cols = _dedupe([c for c in montant_cols if c in df.columns])
    matricule_cols = _dedupe([c for c in matricule_cols if c in df.columns])

    if not in_place:
        df = df.copy()

    # --- MONTANT ---
    if montant_cols:
        subm = df[montant_cols].copy()

        # Nettoie colonnes texte: trim → NaN si vide
        for c in montant_cols:
            if is_string_dtype(subm[c]) or subm[c].dtype == "object":
                s = subm[c].astype("string").str.strip()
                subm[c] = s.replace("", pd.NA)

        # Prend la 1ère non nulle selon la priorité
        df[montant_out] = subm.bfill(axis=1).iloc[:, 0]

        # Convertit en numérique
        df[montant_out] = pd.to_numeric(df[montant_out], errors="coerce")
    else:
        df[montant_out] = pd.NA  # aucune colonne disponible

    # --- MATRICULE ---
    if matricule_cols:
        subx = df[matricule_cols].copy()

        for c in matricule_cols:
            if is_string_dtype(subx[c]) or subx[c].dtype == "object":
                s = subx[c].astype("string").str.strip()
                subx[c] = s.replace("", pd.NA)
            else:
                # Non-texte (int, float...) → garde tel quel, NaN reste NaN
                pass

        df[matricule_out] = subx.bfill(axis=1).iloc[:, 0]
    else:
        df[matricule_out] = pd.NA

    return df

In [16]:
def create_montant_et_matricule_fast(df, montant_cols, matricule_cols, montant_out="MONTANT", matricule_out="MATRICULE_UNIQUE"):
    # --- Montant ---
    sub_montant = df[montant_cols].astype("string").apply(lambda x: x.str.strip())
    sub_montant = sub_montant.mask(sub_montant == "")
    df[montant_out] = pd.to_numeric(sub_montant.stack().groupby(level=0).first(), errors='coerce')

    # --- Matricule ---
    sub_mat = df[matricule_cols].astype("string").apply(lambda x: x.str.strip())
    sub_mat = sub_mat.mask(sub_mat == "")
    df[matricule_out] = sub_mat.stack().groupby(level=0).first()

    return df


In [17]:
def rename_columns_from_mapping(
    df_panel: pd.DataFrame,
    df_mapping: pd.DataFrame,
    codes_col: str = "CODES_POSSIBLES",
    norm_col: str = "LIBELLÉ_POSTE_NORM",
    make_unique: bool = False,
    sep: str = ','
) -> pd.DataFrame:
    """
    Renomme les colonnes de df_panel selon le mapping codes possibles -> libellé normalisé.
    Signale :
      - les colonnes correspondant à des codes mais sans libellé dans le mapping,
      - les colonnes qui ne sont pas des codes et donc non renommées.
    """
    if norm_col not in df_mapping.columns:
        raise KeyError(f"La colonne {norm_col} est absente du mapping.")

    # Colonnes qui ne sont jamais des codes
    exceptions = ['SOURCE_FICHIER', 'MONTANT BRUT', 'MATRICULE||CODE_ORGANISME']

    # Construire dictionnaire code -> libellé normalisé
    map_code_to_label = {}
    for _, row in df_mapping.iterrows():
        codes_raw = str(row[codes_col]) if pd.notna(row[codes_col]) else ''
        codes_list = [c.strip() for c in codes_raw.split(sep) if c.strip()]
        label = str(row[norm_col]).strip()
        for code in codes_list:
            map_code_to_label[code] = label

    # Renommer les colonnes
    old_cols = [str(c).strip() for c in df_panel.columns]
    renamed_base = [map_code_to_label.get(c, c) for c in old_cols]

    # Gestion collisions si make_unique=True
    if make_unique:
        counts = Counter(renamed_base)
        final_cols = [
            f"{new_lab} (code {orig_code})" if counts[new_lab] > 1 and new_lab != orig_code else new_lab
            for orig_code, new_lab in zip(old_cols, renamed_base)
        ]
    else:
        final_cols = renamed_base

    # Appliquer le renommage
    df_out = df_panel.copy()
    df_out.columns = final_cols

    nb_renamed = sum(oc != nc for oc, nc in zip(old_cols, final_cols))
    print(f"✔ Colonnes renommées via mapping (libellé normalisé) : {nb_renamed}/{len(old_cols)}")

    # Colonnes correspondant à des codes mais sans libellé
    non_renamed_codes = [c_old for c_old, c_new in zip(old_cols, renamed_base)
                         if c_old == c_new and c_old not in exceptions]

    # Colonnes non mapping (exceptions)
    non_mapping_cols = [c for c in old_cols if c in exceptions]

    if non_renamed_codes:
        print(f"⚠️ Colonnes correspondant à des codes mais sans libellé trouvé ({len(non_renamed_codes)}): {non_renamed_codes}")
    else:
        print("✅ Toutes les colonnes correspondant à des codes ont été renommées correctement.")

    if non_mapping_cols:
        print(f"ℹ️ Colonnes non mapping (exclues volontairement) : {non_mapping_cols}")

    return df_out


# CHARGEMENT DE LA BASE 

In [18]:
# Paramètres
import io
import pandas as pd
import boto3

# Connexion S3
s3_client = boto3.client(
 "s3",
 endpoint_url="http://minio.mon-namespace.svc.cluster.local:80", # service interne K8s
 aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
 aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
 verify=False # à désactiver si SSL auto-signé ; sinon mettre True avec certificat valide
)

# Paramètres
bucket_name = "admindataanstat"
object_key = "Solde/FUSION/fusion_2021_2025_feuille2_COMPLET.parquet"

# Télécharger l'objet depuis S3
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)

# Charger dans un DataFrame depuis le fichier Parquet
panel_solde_df_2 = pd.read_parquet(io.BytesIO(obj['Body'].read()))

# Afficher un aperçu
panel_solde_df_2.head()

,124,125,126,132,135,140,171,195,198,200,...,9Z3,BRUT MONTANT,MATRICULE || CODE_ORGANISME,MATRICULE || ORGANISME,MATRICULE ||CODE_ORGANISME,MATRICULE||CODE_ORGANISME,MATRICULE||ORGANISME,MONTANT BRUT,Total général,source_fichier
0,None,None,None,None,1405115,None,None,None,None,None,...,None,1405115,None,None,None,052140E00,None,None,None,012021.xlsx
1,None,None,None,None,7754900,None,None,None,None,None,...,None,7754900,None,None,None,067605C00,None,None,None,012021.xlsx
2,None,None,None,None,None,None,None,None,None,None,...,None,125000,None,None,None,078350S15,None,None,None,012021.xlsx
3,None,None,None,None,None,None,None,None,None,None,...,None,1312498,None,None,None,081872NAG,None,None,None,012021.xlsx
4,658347,None,None,None,None,None,None,None,None,None,...,None,2380634,None,None,None,084227J15,None,None,None,012021.xlsx


## Information sur la base

In [19]:
print("Le nombre d'observations est :",len(panel_solde_df_2))
print("Le nombre de variables est :",len(panel_solde_df_2.columns))
print("Liste des variables :", list(panel_solde_df_2.columns))

Le nombre d'observations est : 16804107
Le nombre de variables est : 222
Liste des variables : ['124', '125', '126', '132', '135', '140', '171', '195', '198', '200', '201', '202', '203', '204', '207', '208', '21', '210', '211', '213', '214', '216', '221', '223', '224', '225', '226', '228', '229', '230', '231', '232', '235', '236', '237', '238', '239', '240', '241', '242', '243', '244', '245', '248', '249', '251', '253', '257', '258', '259', '260', '261', '262', '263', '264', '265', '268', '270', '271', '272', '275', '276', '278', '280', '281', '286', '287', '288', '293', '294', '295', '296', '297', '299', '310', '311', '312', '313', '320', '321', '322', '323', '324', '330', '335', '336', '340', '341', '342', '343', '344', '345', '351', '352', '353', '368', '369', '370', '371', '372', '373', '374', '375', '376', '377', '380', '381', '382', '390', '394', '402', '403', '404', '406', '410', '411', '412', '413', '415', '416', '417', '418', '419', '420', '421', '422', '423', '424', '425', '4

## 1. Vérification

### Montant

In [20]:
panel_solde_df_2 = panel_solde_df_2.rename(columns={
    "Total général": "TOTAL GENERAL",
    "TOTAL GÉNÉRAL": "TOTAL GENERAL",  # au cas où dans d'autres fichiers
    "source_fichier": "SOURCE_FICHIER"
})

problemes_montant_idx = check_completeness(
    panel_solde_df_2,
    ["BRUT MONTANT", "MONTANT BRUT", "TOTAL GENERAL"],
    title="Montants"
)



📊 Résumé de la vérification [Montants] :
🔹 Nombre total de lignes : 16804107
✅ Lignes correctes (exactement 1 colonne renseignée) : 16804067
⚠️ Lignes à problème (0 ou plusieurs colonnes renseignées) : 40
📌 Index des lignes problématiques : [746540, 746695, 747717, 750357, 760448, 763504, 822582, 833771, 833832, 833843, 833887, 835443, 845232, 858176, 858318, 1254168, 1254301, 1255281, 2026137, 2026263, 2026974, 2027976, 2029109, 2041346, 2065369, 2099683, 2111067, 2111081, 2111128, 2112685, 2135572, 2135713, 4917676, 4917729, 4918167, 5020429, 5023679, 5824140, 5827384, 8585189]


### Matricule

In [21]:
problemes_matricule_idx = check_completeness(
    panel_solde_df_2,
    [
        "MATRICULE || CODE_ORGANISME",
        "MATRICULE || ORGANISME",
        "MATRICULE ||CODE_ORGANISME",
        "MATRICULE||CODE_ORGANISME",
        "MATRICULE||ORGANISME",
    ],
    title="Matricule", 
)



📊 Résumé de la vérification [Matricule] :
🔹 Nombre total de lignes : 16804107
✅ Lignes correctes (exactement 1 colonne renseignée) : 16804107
⚠️ Lignes à problème (0 ou plusieurs colonnes renseignées) : 0
📌 Index des lignes problématiques : []


## 2. Suppression des lignes inutiles 

In [ ]:
panel_solde_df_2 = drop_if_other_cols_empty_ultralight(
    df=panel_solde_df_2,
    index_to_check=problemes_montant_idx,
    keep_cols=[
        "MATRICULE || CODE_ORGANISME",
        "MATRICULE || ORGANISME",
        "MATRICULE ||CODE_ORGANISME",
        "MATRICULE||CODE_ORGANISME",
        "MATRICULE||ORGANISME",
        "source_fichier"
    ],
    title="Montants"
)

print("Nombre total de lignes après nettoyage :", len(panel_solde_df_2))


## 3. Création des variables Montant et matricule 

In [22]:
panel_solde_df_2 = create_montant_et_matricule_fast(
    panel_solde_df_2,
    montant_cols=["BRUT MONTANT", "MONTANT BRUT", "TOTAL GENERAL"],
    matricule_cols=[
        "MATRICULE||CODE_ORGANISME",
        "MATRICULE || CODE_ORGANISME",
        "MATRICULE ||CODE_ORGANISME",
        "MATRICULE||ORGANISME",
        "MATRICULE || ORGANISME",
    ],
    montant_out="MONTANT",
    matricule_out="MATRICULE_UNIQUE"
)


In [24]:
# Lister les variables de la base de données pour vérification
panel_solde_df_2.columns

Index(['124', '125', '126', '132', '135', '140', '171', '195', '198', '200',
       ...
       'MATRICULE || CODE_ORGANISME', 'MATRICULE || ORGANISME',
       'MATRICULE ||CODE_ORGANISME', 'MATRICULE||CODE_ORGANISME',
       'MATRICULE||ORGANISME', 'MONTANT BRUT', 'TOTAL GENERAL',
       'SOURCE_FICHIER', 'MONTANT', 'MATRICULE_UNIQUE'],
      dtype='object', length=224)

In [25]:
# Pour les colonnes numériques
desc = panel_solde_df_2['MONTANT'].describe()
desc['missing'] = panel_solde_df_2['MONTANT'].isna().sum()
print(desc)

# Pour les colonnes texte
des = panel_solde_df_2['MATRICULE_UNIQUE'].describe()
des['missing'] = panel_solde_df_2['MATRICULE_UNIQUE'].isna().sum()
print(des)

count         16804067.0
mean       479248.244324
std        452671.989157
min                  0.0
25%             305752.0
50%             398883.0
75%             528538.0
max           62781031.0
missing             40.0
Name: MONTANT, dtype: Float64
count       16804107
unique        405279
top        803079A12
freq              58
missing            0
Name: MATRICULE_UNIQUE, dtype: object


In [ ]:
panel_solde_df_2 = panel_solde_df_2.drop(
    columns=[
        "BRUT MONTANT",
        "MONTANT BRUT",
        "TOTAL GENERAL",
        "MATRICULE||CODE_ORGANISME",
        "MATRICULE || CODE_ORGANISME",
        "MATRICULE ||CODE_ORGANISME",
        "MATRICULE||ORGANISME",
        "MATRICULE || ORGANISME",
    ]
)


In [29]:
# Renommer des variables
panel_solde_df_2=panel_solde_df_2.rename(
    columns={"MONTANT" : "MONTANT BRUT", 
             "MATRICULE_UNIQUE" : "MATRICULE||CODE_ORGANISME"
            }
)

## 4. Chargement du fichier avec les codes 

In [26]:
# Paramètres S3
bucket_name = "admindataanstat"
object_key = "Solde/FICHIER_ANSTAT_CODE_2025.xlsx"  

# Télécharger le fichier depuis S3 dans un buffer mémoire
obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)
bytes_io = io.BytesIO(obj['Body'].read())

# Remettre le curseur au début
bytes_io.seek(0)

xlsx = pd.ExcelFile(bytes_io)
print("Feuilles disponibles :", xlsx.sheet_names)

Feuilles disponibles : ['lieu affectation', 'ORGANISME_OK', 'CODE_SITUATION_SOLDE', 'HISTORIQUE_ECHELLES_CORPS', 'ECHELLES_CORPS_ACTUEL', 'service', 'FONCTION', 'grade', 'LIBELLE POSTE']


In [27]:
# Lire la feuille POSTE dans notre fichier Excel de référence
fichier_ref = "LIBELLE POSTE"
df_ref = read_reference(bytes_io, fichier_ref)


=== LIBELLE POSTE | APERÇU BRUT (top 5) ===


,CODE_POSTE,LIBELLÉ_POSTE
0,---,pour epn
1,112,Traitement Enseignant C H U
2,124,Solde de Base
3,125,Salaire d'Agent Temporaire
4,126,Salaire de Gens de Maison


Colonnes BRUTES (LIBELLE POSTE) : ['CODE_POSTE', 'LIBELLÉ_POSTE']
Shape brut : (588, 2)

=== LIBELLE POSTE | APRÈS NETTOYAGE (top 5) ===


,CODE_POSTE,LIBELLÉ_POSTE
0,---,pour epn
1,112,Traitement Enseignant C H U
2,124,Solde de Base
3,125,Salaire d'Agent Temporaire
4,126,Salaire de Gens de Maison


Colonnes NETTOYÉES (LIBELLE POSTE) : ['CODE_POSTE', 'LIBELLÉ_POSTE']
Shape nettoyé : (588, 2)


### 4.1. Définition des paramètres

In [31]:
nm_feuille = "POSTE" # nom de la feuille Excel à charger
var_code_ext = "CODE_POSTE" # la colonne de code dans la table externe (fichier excel)
var_lib_ext  = "LIBELLÉ_POSTE" #la colonne de libellé dans la table externe (fichier Excel)
var_cle_ext  = var_lib_ext # on choisit le libellé comme clé principale côté référence (fichier Excel)

# Noms normalisés
var_norm_ext   = f"{var_cle_ext}_NORM"  # nom de la variable de libellé normalisé dans la table de référence (fichier Excel)

### 4.2. Vérification

In [32]:
# Contrôle qualité de la table de référence
_ = check_code_to_label_uniqueness(df_ref, var_code_ext, var_lib_ext) # Vérifie que chaque code (var_code_ext) correspond à un seul libellé (var_lib_ext) dans le fichier Excel

# Crée (si elle n’existe pas déjà) une colonne var_norm_ext dans df_ref (fichier excel)
var_norm_ext = ensure_normalized_column(df_ref, var_lib_ext, var_norm_ext)

# Vérifie qu’il n’y a pas de doublons après normalisation
_ = detect_normalized_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)

✅ Chaque code est associé à un seul libellé.
⚠️ Doublons détectés après normalisation : 12 clés répétées, 28 lignes uniques concernées.


In [33]:
# Afficher la liste exhaustive des doublons après normalisation
df_dups = report_duplicates(df_ref, var_norm_ext, var_lib_ext, var_code_ext)

⚠️ Doublons détectés après normalisation : 12 clés répétées, 28 lignes concernées.


,CODE_POSTE,LIBELLÉ_POSTE,LIBELLÉ_POSTE_NORM
39,221,Différentiel Familial,differentiel familial
81,263,Indemnité de Logement,indemnite de logement
83,265,Indemnité de Logement,indemnite de logement
89,271,Indemnité de Logement,indemnite de logement
178,422,Différentiel Familial,differentiel familial
196,440,Participation judicature,participation judicature
197,441,Participation judicature,participation judicature
198,442,Participation judicature,participation judicature
199,443,Participation judicature,participation judicature
243,487,Responsabilité Trésor,responsabilite tresor


### 4.3. Mapping aléatoire (1 code / clé) + export dans un classeur commun

In [34]:
# Construction du mapping
mapping = build_random_mapping(df_ref, var_norm_ext, var_code_ext, var_lib_ext, seed=123)

# Export du mapping dans un fichier Excel 
export_mapping(mapping, workbook_path="Referentiels_unifies.xlsx", sheet_name= nm_feuille)

/tmp/ipykernel_790/2839350002.py:31: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=1, random_state=seed))


✅ Feuille écrite : POSTE → Referentiels_unifies.xlsx


### 4.4. Fonction pour le mapping

In [35]:
panel_solde_df_2 = rename_columns_from_mapping(
    df_panel=panel_solde_df_2,
    df_mapping=mapping,  
    codes_col="CODES_POSSIBLES",
    norm_col="LIBELLÉ_POSTE_NORM",
    make_unique=True
)

✔ Colonnes renommées via mapping (libellé normalisé) : 187/190
✅ Toutes les colonnes correspondant à des codes ont été renommées correctement.
ℹ️ Colonnes non mapping (exclues volontairement) : ['SOURCE_FICHIER', 'MONTANT BRUT', 'MATRICULE||CODE_ORGANISME']


In [36]:
list(panel_solde_df_2.columns)

['solde de base',
 'salaire d agent temporaire',
 'salaire de gens de maison',
 'forfait indiciaire ambassadeur',
 'compensatrice smig',
 'revalorisation 2014',
 'compensatrice non imp ambassades',
 'risque sanitaire',
 'compensatrice non imposable',
 'compensatrice imposable',
 'complement inp hb',
 'sujetion greffier en chef',
 'prime de recherche',
 'compensatrice ambassadeur',
 'contrainte',
 'heures suppl , personnel enseignant',
 'heures de cours',
 'patrouille',
 'traitement personnel non classe',
 'heures supplementaires administration',
 'forfait cap',
 'differentiel familial (code 221)',
 'traitement chef de canton',
 'utilisation vehicule p/ppnat',
 'risque',
 'verification chambre cptes/poincon or',
 'transport',
 'prime de caisse',
 'entretien uniforme',
 'acquisition uniforme',
 'deplacement temporaire',
 'allocation eleve',
 'traitement ambassadeur',
 'indemnite ingenieur systeme',
 'regularisation cn exercice anterieur',
 'regularisation igr exercice anterieur',
 'regul

# SAUVEGARDE 

## SAUVEGARDE DU FICHIER DES CODES DEFINITIFS

In [127]:
# Connexion S3 / MinIO
s3_client = boto3.client(
    "s3",
    endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
    aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
    aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
    verify=False
)

bucket_name = "admindataanstat"
object_key = "Solde/Referentiels_unifies.xlsx"

# Lecture du fichier local ou en mémoire contenant toutes les feuilles
excel_path_local = "Referentiels_unifies.xlsx"  # ou un chemin local existant
all_sheets = pd.read_excel(excel_path_local, sheet_name=None)  # sheet_name=None -> toutes les feuilles

# Conversion en Excel en mémoire
excel_buffer = io.BytesIO()
with pd.ExcelWriter(excel_buffer, engine="openpyxl") as writer:
    for sheet_name, df in all_sheets.items():
        df.to_excel(writer, index=False, sheet_name=sheet_name)

# Upload sur S3
s3_client.put_object(Bucket=bucket_name, Key=object_key, Body=excel_buffer.getvalue())

print(f"✅ Toutes les feuilles exportées sur S3 : s3://{bucket_name}/{object_key}")


✅ Toutes les feuilles exportées sur S3 : s3://admindataanstat/Solde/Referentiels_unifies.xlsx


## SAUVEGARDE DE LA BASE DES FEUILLES 2 AVEC COLONNES RENOMMEES

In [37]:
# Connexion S3 / MinIO
s3_client = boto3.client(
    "s3",
    endpoint_url="http://minio.mon-namespace.svc.cluster.local:80",
    aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
    aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
    verify=False
)

bucket_name = "admindataanstat"
object_key = "Solde/FUSION/fusion_2_2021_2025_final.parquet"

# Sauvegarde dans un buffer mémoire
buffer = io.BytesIO()
panel_solde_df_2.to_parquet(buffer, engine='pyarrow', index=False)
buffer.seek(0)  # Repositionne au début du buffer

# Upload sur MinIO
s3_client.put_object(Bucket=bucket_name, Key=object_key, Body=buffer)

print(f"✅ Base sauvegardée en Parquet sur MinIO : s3://{bucket_name}/{object_key}")

✅ Base sauvegardée en Parquet sur MinIO : s3://admindataanstat/Solde/panel_solde_df_2.parquet


# II. JOINTURE DE fusion_1_2021_2025_final.parquet ET fusion_2_2021_2025_final.parquet

## CHARGEMENT DES PACKAGES NÉCESSAIRES

In [ ]:
import io
import re
import gc
import os
import math
import time
from io import BytesIO
from typing import Optional, Dict, List, Iterable

import numpy as np
import pandas as pd

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

# Limite l'utilisation CPU des libs natives -> plus stable en RAM
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")



## 2) DÉFINITION DES FONCTIONS

### 2.1 Lecture S3 : parquet -> DataFrame

In [ ]:
def read_parquet_from_s3(
    bucket: str,
    key: str,
    *,
    endpoint_url: str,
    aws_access_key_id: str,
    aws_secret_access_key: str,
    verify: bool = False,
    columns: List[str] | None = None,
    engine: str | None = "pyarrow",
    max_attempts: int = 6,
    base_sleep: float = 1.0,
    fallback_download: bool = True
) -> pd.DataFrame:
    """
    Lit un Parquet depuis S3/MinIO en gérant les ralentissements 'SlowDown'.
    """
    s3_client = boto3.client(
        "s3",
        endpoint_url=endpoint_url,
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        verify=verify,
        config=Config(
            retries={"max_attempts": max_attempts, "mode": "adaptive"},
            max_pool_connections=16,
            read_timeout=180,
            connect_timeout=45,
        ),
    )

    attempt = 0
    while True:
        try:
            obj = s3_client.get_object(Bucket=bucket, Key=key)
            return pd.read_parquet(io.BytesIO(obj["Body"].read()), columns=columns, engine=engine)
        except ClientError as e:
            code = e.response.get("Error", {}).get("Code", "")
            retryable = code in {"SlowDown", "SlowDownRead", "RequestTimeout", "503", "503SlowDown"}
            attempt += 1
            if retryable and attempt < max_attempts:
                sleep_s = min(base_sleep * (2 ** (attempt - 1)), 30)
                print(f"⚠️ {code}: tentative {attempt}/{max_attempts} — nouvelle tentative dans ~{sleep_s:.1f}s…")
                time.sleep(sleep_s)
                continue
            if fallback_download:
                print(f"ℹ️ Passage en mode 'download then read' (raison: {code or 'échec lecture stream'})")
                obj = s3_client.get_object(Bucket=bucket, Key=key)
                return pd.read_parquet(io.BytesIO(obj["Body"].read()), columns=columns, engine=engine)
            raise



### 2.2 Détection de bases "répétées"

In [ ]:
def _base_name(col: str) -> str:
    s = str(col).strip()
    s = re.sub(r"\s*\([^)]*\)\s*$", "", s)
    s = re.sub(r"[_\-|./]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s.lower()

def find_repeated_columns(df: pd.DataFrame, exclude: Iterable[str] = ("SOURCE_FICHIER",)) -> Dict[str, List[str]]:
    groups: Dict[str, List[str]] = {}
    for c in df.columns:
        if c in exclude:
            continue
        base = _base_name(c)
        groups.setdefault(base, []).append(c)
    return {base: cols for base, cols in groups.items() if len(cols) > 1}

def report_repeated_columns(df: pd.DataFrame, exclude: Iterable[str] = ("SOURCE_FICHIER",)) -> pd.DataFrame:
    rep = find_repeated_columns(df, exclude=exclude)
    if not rep:
        print("✅ Aucune colonne répétée détectée.")
        return pd.DataFrame(columns=["base", "nb_colonnes", "colonnes"])
    print(f"🔎 Bases répétées détectées: {len(rep)}")
    rows = []
    for base, cols in sorted(rep.items()):
        print(f"  • {base} -> {cols}")
        rows.append({"base": base, "nb_colonnes": len(cols), "colonnes": cols})
    return (pd.DataFrame(rows)
              .sort_values(["nb_colonnes", "base"], ascending=[False, True])
              .reset_index(drop=True))




### 2.4 Consolidation haute performance

In [ ]:

def _filled_mask(block: pd.DataFrame, treat_zero_as_filled: bool) -> np.ndarray:
    n, m = block.shape
    filled = np.zeros((n, m), dtype=bool)
    for j, col in enumerate(block.columns):
        s = block[col]
        if pd.api.types.is_numeric_dtype(s):
            ok = s.notna().to_numpy()
            if not treat_zero_as_filled:
                ok &= (s.to_numpy() != 0)
        else:
            v = s.astype("object").to_numpy()
            ok = pd.notna(v)
            tmp = np.empty_like(v, dtype=object)
            if ok.any():
                tmp[ok] = np.char.lower(np.char.strip(v[ok].astype(str)))
                bad_ok = (tmp[ok] == "") | (tmp[ok] == "na") | (tmp[ok] == "nan") | (tmp[ok] == "none")
                bad = np.zeros(ok.shape, dtype=bool)
                bad[ok] = bad_ok
                ok &= ~bad
        filled[:, j] = ok
    return filled

def _coalesce_first_non_null(block: pd.DataFrame, treat_zero_as_filled: bool) -> pd.Series:
    if block.shape[1] == 1:
        s = block.iloc[:, 0]
        if pd.api.types.is_numeric_dtype(s) and not treat_zero_as_filled:
            return s.where(s.notna() & (s != 0))
        return s
    filled = _filled_mask(block, treat_zero_as_filled)
    any_filled = filled.any(axis=1)
    first_idx = filled.argmax(axis=1)
    arr = block.to_numpy(copy=False)
    rows = np.arange(arr.shape[0])
    out_vals = np.empty(arr.shape[0], dtype=object)
    out_vals[:] = pd.NA
    out_vals[any_filled] = arr[rows[any_filled], first_idx[any_filled]]
    return pd.Series(out_vals, index=block.index, dtype="object")

def _sum_numeric_block(block: pd.DataFrame, chunk: int = 50) -> pd.Series:
    """Somme ligne par ligne en traitant par paquets de colonnes"""
    idx = block.index
    out = pd.Series(0.0, index=idx, dtype="float64")
    ncols = block.shape[1]
    start = 0
    while start < ncols:
        subcols = block.columns[start:start+chunk]
        to_sum = {c: pd.to_numeric(block[c], errors="coerce") for c in subcols}
        part = pd.DataFrame(to_sum, index=idx).sum(axis=1, min_count=1).fillna(0.0)
        out = out.add(part, fill_value=0.0)
        start += chunk
    out = out.mask(out == 0.0, other=np.nan)
    return out

def _concat_block(block: pd.DataFrame, sep: str = " | ") -> pd.Series:
    def _row_concat(row):
        vals = [str(v) for v in row if pd.notna(v) and str(v).strip() not in ("", "na", "nan", "none")]
        return sep.join(vals) if vals else pd.NA
    return block.apply(_row_concat, axis=1)

def consolidate_all_years_and_drop_sources(
    df: pd.DataFrame,
    source_col: str = "SOURCE_FICHIER",
    *,
    mode: str = "first_non_null",
    sep: str = " | ",
    treat_zero_as_filled: bool = True,
    new_name_fmt: str = "{base}",
    sum_chunk: int = 50
) -> pd.DataFrame:
    """Consolide les colonnes répétées et supprime les sources"""
    if source_col not in df.columns:
        raise ValueError(f"Colonne {source_col!r} absente.")
    out = df.copy()

    repeated = find_repeated_columns(out, exclude=(source_col,))
    if not repeated:
        print("ℹ️ Aucune base répétée. Rien à consolider.")
        return out

    years = extract_years_unique(out[source_col])

    def _apply_mode(block: pd.DataFrame) -> pd.Series:
        if mode == "sum":
            return _sum_numeric_block(block, chunk=sum_chunk)
        elif mode == "first_non_null":
            return _coalesce_first_non_null(block, treat_zero_as_filled)
        elif mode == "concat":
            return _concat_block(block, sep=sep)
        else:
            raise ValueError(f"mode inconnu: {mode!r}")

    if years:
        out["_YEAR_"] = extract_years_series(out[source_col])
        for base, cols in repeated.items():
            new_col = new_name_fmt.format(base=base)
            if new_col not in out.columns:
                out[new_col] = pd.NA
            for y in years:
                mask = out["_YEAR_"].eq(y)
                if not mask.any():
                    continue
                block = out.loc[mask, cols]
                out.loc[mask, new_col] = _apply_mode(block)
        out.drop(columns=["_YEAR_"], inplace=True, errors="ignore")
    else:
        for base, cols in repeated.items():
            new_col = new_name_fmt.format(base=base)
            out[new_col] = _apply_mode(out[cols])

    cols_to_drop = sorted({c for cols in repeated.values() for c in cols})
    out.drop(columns=cols_to_drop, inplace=True, errors="ignore")
    gc.collect()
    return out



### 2.5 Vérification d'exclusivité

In [ ]:
def check_repeated_columns_exclusivity_by_year(
    df: pd.DataFrame,
    *,
    source_col: str = "SOURCE_FICHIER",
    treat_zero_as_filled: bool = True,
    show_examples: int = 10
) -> pd.DataFrame:
    if source_col not in df.columns:
        raise KeyError(f"Colonne {source_col!r} absente.")

    repeated = find_repeated_columns(df, exclude=(source_col,))
    if not repeated:
        print("✅ Aucune base répétée détectée — rien à vérifier.")
        return pd.DataFrame(columns=["base", "year", "violations", "rows_in_year", "columns"])

    years_series = extract_years_series(df[source_col])
    if years_series.isna().all():
        print("⚠️ Aucune année détectée — contrôle global.")
        years_series = pd.Series(["ALL"] * len(df), index=df.index)

    id_cols = [c for c in ["MATRICULE", "MATRICULE||CODE_ORGANISME", source_col] if c in df.columns] or [source_col]
    rows_summary = []
    any_violation = False

    for base, cols in repeated.items():
        block = df[cols]
        filled = _filled_mask(block, treat_zero_as_filled)
        nb_filled = filled.sum(axis=1)

        for year_val, inds in years_series.groupby(years_series).groups.items():
            mask_year = df.index.isin(inds)
            n_in_year = int(mask_year.sum())
            if n_in_year == 0:
                continue

            viol_mask = (nb_filled > 1) & mask_year
            n_viol = int(viol_mask.sum())

            rows_summary.append({
                "base": base,
                "year": year_val,
                "violations": n_viol,
                "rows_in_year": n_in_year,
                "columns": cols
            })

            if n_viol > 0:
                any_violation = True
                print(f"\n🚨 Exclusivité violée — base='{base}', année={year_val} : {n_viol} lignes en anomalie (sur {n_in_year}).")
                ex_cols = list(dict.fromkeys(id_cols + cols))
                print(df.loc[viol_mask, ex_cols].head(show_examples).to_string(index=False))

    summary = (pd.DataFrame(rows_summary)
                 .sort_values(["violations", "base", "year"], ascending=[False, True, True])
                 .reset_index(drop=True))

    if not any_violation:
        print("✅ Exclusivité respectée : au plus une colonne renseignée par base/année.")
    else:
        total_viol = int(summary["violations"].sum()) if not summary.empty else 0
        print(f"\nBilan: {total_viol} violations au total.")

    return summary



### 2.6 Fonctions pour la fusion

In [ ]:

def write_df_to_xlsx_chunked(
    df: pd.DataFrame,
    path: str,
    *,
    sheet_base: str = "data",
    chunk_rows: int = 1_000_000,
    engine: str = "xlsxwriter"
) -> None:
    n = len(df)
    if n == 0:
        with pd.ExcelWriter(path, engine=engine) as xw:
            df.to_excel(xw, sheet_name=sheet_base, index=False)
        return
    parts = math.ceil(n / chunk_rows)
    with pd.ExcelWriter(path, engine=engine) as xw:
        for i in range(parts):
            start = i * chunk_rows
            stop  = min((i + 1) * chunk_rows, n)
            sheet_name = sheet_base if i == 0 else f"{sheet_base}_part{i+1}"
            df.iloc[start:stop].to_excel(xw, sheet_name=sheet_name, index=False)

def _year_month_from_source_file(s: pd.Series) -> pd.DataFrame:
    ss = s.astype("string").str.strip()
    m = ss.str.extract(r'^\s*(0[1-9]|1[0-2])\s*((?:19|20)\d{2})', expand=True)
    return pd.DataFrame({
        "YEAR":  pd.to_numeric(m[1], errors="coerce").astype("Int64"),
        "MONTH": pd.to_numeric(m[0], errors="coerce").astype("Int64"),
    })

def _year_month_from_date_collecte(s: pd.Series) -> pd.DataFrame:
    dt = pd.to_datetime(s, errors="coerce", utc=False)
    return pd.DataFrame({
        "YEAR":  dt.dt.year.astype("Int64"),
        "MONTH": dt.dt.month.astype("Int64"),
    })

def _normalize_matricule_min(s: pd.Series) -> pd.Series:
    return (s.astype("string")
             .str.strip()
             .str.replace("\u00A0", "", regex=False))

def merge_union_for_year_with_suffix(
    df_left: pd.DataFrame,
    df_right: pd.DataFrame,
    year: int,
    *,
    join_on_month: bool = True,
    suffix_left: str = "_b0",
    suffix_right: str = "_b1",
    dedup_left: bool = False,
    dedup_right: bool = True,
    export_unmatched_prefix: str | None = None,
    xlsx_chunk_rows: int = 1_000_000,
    xlsx_engine: str = "xlsxwriter",
    keep_left: List[str] | None = None,
    keep_right: List[str] | None = None,
    max_xlsx_rows: int = 200_000
) -> pd.DataFrame:
    
    # Copies + normalisation matricule
    L = df_left.copy()
    R = df_right.copy()
    for d in (L, R):
        if "MATRICULE||CODE_ORGANISME" not in d.columns:
            raise KeyError("Il manque 'MATRICULE||CODE_ORGANISME' dans une des bases.")
        d["MATRICULE"] = _normalize_matricule_min(d["MATRICULE||CODE_ORGANISME"])

    # YEAR/MONTH
    if "SOURCE_FICHIER" not in L.columns:
        raise KeyError("La base gauche doit contenir 'SOURCE_FICHIER'.")
    ymL = _year_month_from_source_file(L["SOURCE_FICHIER"])
    L["YEAR"], L["MONTH"] = ymL["YEAR"], ymL["MONTH"]

    if "DATE_COLLECTE" not in R.columns:
        raise KeyError("La base droite doit contenir 'DATE_COLLECTE'.")
    ymR = _year_month_from_date_collecte(R["DATE_COLLECTE"])
    R["YEAR"], R["MONTH"] = ymR["YEAR"], ymR["MONTH"]

    # Filtre année
    L = L.loc[L["YEAR"].eq(year)].copy()
    R = R.loc[R["YEAR"].eq(year)].copy()

    # Clés + projection
    keys = (["YEAR", "MONTH", "MATRICULE"] if join_on_month else ["YEAR", "MATRICULE"])

    if keep_left is not None:
        selL = list(set(keys + keep_left))
        L = L.loc[:, [c for c in selL if c in L.columns]]

    if keep_right is not None:
        selR = list(set(keys + keep_right))
        R = R.loc[:, [c for c in selR if c in R.columns]]

    # Colonnes communes -> suffixes
    overlap = sorted((set(L.columns) & set(R.columns)) - set(keys))
    if overlap:
        L.rename(columns={c: f"{c}{suffix_left}"  for c in overlap}, inplace=True)
        R.rename(columns={c: f"{c}{suffix_right}" for c in overlap}, inplace=True)

    # Dédup
    if dedup_left:
        L = L.sort_values(keys).drop_duplicates(keys, keep="first")
    if dedup_right:
        R = R.sort_values(keys).drop_duplicates(keys, keep="first")

    # Convertir texte en catégorie
    for d in (L, R):
        for c in d.select_dtypes(include=["object", "string"]).columns:
            if c not in keys:
                try:
                    d[c] = d[c].astype("category")
                except Exception:
                    pass

    # OUTER JOIN
    merged = L.merge(R, on=keys, how="outer", suffixes=("", ""), indicator=True)

    # Indicateurs
    src_map = {"left_only": "base1", "right_only": "base2", "both": "both"}
    merged["IN_SRC"] = merged["_merge"].map(src_map)
    merged["IN_BASE1"] = merged["_merge"].isin(["left_only", "both"])
    merged["IN_BASE2"] = merged["_merge"].isin(["right_only", "both"])

    print(f"Fusion OUTER {year} (join_on_month={join_on_month}) → {len(merged):,} lignes, {merged.shape[1]} colonnes")
    print("\nBilan _merge :")
    print(merged["_merge"].value_counts(dropna=False))

    # Exports non-appariés
    if export_unmatched_prefix:
        left_cols  = [c for c in ["YEAR","MONTH","MATRICULE","SOURCE_FICHIER"] if c in merged.columns]
        right_cols = [c for c in ["YEAR","MONTH","MATRICULE","DATE_COLLECTE"] if c in merged.columns]

        left_only_df  = merged.loc[merged["_merge"]=="left_only",  left_cols]
        right_only_df = merged.loc[merged["_merge"]=="right_only", right_cols]

        if len(left_only_df) > 0:
            if len(left_only_df) <= max_xlsx_rows:
                left_path = f"{export_unmatched_prefix}_{year}_left_only.xlsx"
                write_df_to_xlsx_chunked(left_only_df, left_path, sheet_base="left_only",
                                         chunk_rows=min(xlsx_chunk_rows, max(10_000, len(left_only_df))),
                                         engine=xlsx_engine)
                print("XLSX écrit :", left_path)
            else:
                left_path = f"{export_unmatched_prefix}_{year}_left_only.parquet"
                left_only_df.to_parquet(left_path, engine="pyarrow", index=False)
                print("Parquet écrit (plus léger) :", left_path)

        if len(right_only_df) > 0:
            if len(right_only_df) <= max_xlsx_rows:
                right_path = f"{export_unmatched_prefix}_{year}_right_only.xlsx"
                write_df_to_xlsx_chunked(right_only_df, right_path, sheet_base="right_only",
                                         chunk_rows=min(xlsx_chunk_rows, max(10_000, len(right_only_df))),
                                         engine=xlsx_engine)
                print("XLSX écrit :", right_path)
            else:
                right_path = f"{export_unmatched_prefix}_{year}_right_only.parquet"
                right_only_df.to_parquet(right_path, engine="pyarrow", index=False)
                print("Parquet écrit (plus léger) :", right_path)

    gc.collect()
    return merged



### 2.7 Vérification somme

In [ ]:
def get_multi_filled_row_indices(
    df: pd.DataFrame,
    *,
    source_col: str = "SOURCE_FICHIER",
    treat_zero_as_filled: bool = True,
) -> pd.Index:
    if source_col not in df.columns:
        raise KeyError(f"Colonne {source_col!r} absente.")

    repeated = find_repeated_columns(df, exclude=(source_col,))
    if not repeated:
        return pd.Index([])

    years_series = extract_years_series(df[source_col])
    if years_series.isna().all():
        years_series = pd.Series(["ALL"] * len(df), index=df.index)

    bad_idx_sets: List[pd.Index] = []
    for base, cols in repeated.items():
        block = df[cols]
        filled = _filled_mask(block, treat_zero_as_filled)
        nb_filled = filled.sum(axis=1)

        for _, inds in years_series.groupby(years_series).groups.items():
            mask_year = df.index.isin(inds)
            viol_mask = (nb_filled > 1) & mask_year
            if viol_mask.any():
                bad_idx_sets.append(df.index[viol_mask])

    if not bad_idx_sets:
        return pd.Index([])

    bad_union = bad_idx_sets[0]
    for idx in bad_idx_sets[1:]:
        bad_union = bad_union.union(idx)
    return bad_union

def check_sum_all_base_vs_reference_on_index(
    df: pd.DataFrame,
    row_index_filter: pd.Index,
    *,
    reference_col: str = "MONTANT BRUT",
    abs_tol: float = 1e-6,
    rel_tol: float = 0.0,
    show_examples: int = 10,
) -> pd.DataFrame:
    if reference_col not in df.columns:
        raise KeyError(f"Colonne de référence {reference_col!r} absente.")

    if len(row_index_filter) == 0:
        print("✅ Aucune ligne à tester (aucune exclusivité violée).")
        return pd.DataFrame(columns=["sum_all", "reference", "abs_diff", "rel_diff"])

    sub = df.loc[row_index_filter]

    num_cols = sub.select_dtypes(include=["number"]).columns.tolist()
    ref_numeric = (pd.to_numeric(sub[reference_col], errors="coerce")
                   if reference_col not in num_cols else sub[reference_col])

    sum_all = sub[num_cols].sum(axis=1, min_count=1)
    abs_diff = (sum_all - ref_numeric).abs()
    rel_diff = abs_diff / ref_numeric.replace(0, pd.NA)
    bad_mask = (abs_diff > abs_tol) & (rel_diff.fillna(np.inf) > rel_tol)

    out = sub.copy()
    out["sum_all"] = sum_all
    out["reference"] = ref_numeric
    out["abs_diff"] = abs_diff
    out["rel_diff"] = rel_diff

    anomalies = out.loc[bad_mask].sort_values("abs_diff", ascending=False)
    print(f"🔎 Lignes testées: {len(sub):,} | Anomalies (somme ≠ {reference_col}): {len(anomalies):,}")
    if not anomalies.empty:
        print("\n📌 Exemples d'anomalies:")
        print(anomalies.head(show_examples).to_string(index=True))
    return anomalies



### 2.8 Enregistremen S3

In [ ]:
def get_s3_client(endpoint_url, aws_key, aws_secret, verify_ssl):
    return boto3.client(
        "s3",
        endpoint_url=endpoint_url,
        aws_access_key_id=aws_key,
        aws_secret_access_key=aws_secret,
        verify=verify_ssl,
    )

def save_parquet_to_s3(df, object_key: str, bucket: str, s3_client, engine: str = "pyarrow"):
    buf = io.BytesIO()
    df.to_parquet(buf, engine=engine, index=False)
    buf.seek(0)
    s3_client.put_object(Bucket=bucket, Key=object_key, Body=buf)
    print(f"✅ Sauvegardé : s3://{bucket}/{object_key}")
    return f"s3://{bucket}/{object_key}"

def save_panel_solde_year_parquet(df, year: int, prefix: str, bucket: str, s3_client):
    key = f"{prefix}{year}.parquet"
    return save_parquet_to_s3(df, object_key=key, bucket=bucket, s3_client=s3_client)



### 2.3 Extraction d'années

In [ ]:

 def extract_years_series(s: pd.Series) -> pd.Series:
    """Extrait l'année depuis SOURCE_FICHIER (format: MMYYYY.xlsx)"""
    ss = s.astype("string").str.strip()
    m = ss.str.extract(r'((?:19|20)\d{2})', expand=False)
    return pd.to_numeric(m, errors="coerce").astype("Int64")

def extract_years_unique(s: pd.Series) -> List[int]:
    """Retourne la liste unique des années détectées"""
    years = extract_years_series(s).dropna().unique()
    return sorted([int(y) for y in years])


### 2.9 Empilement

In [ ]:
def read_parquet_s3(object_key: str, bucket: str, s3_client) -> pd.DataFrame:
    obj = s3_client.get_object(Bucket=bucket, Key=object_key)
    return pd.read_parquet(BytesIO(obj["Body"].read()))

def stack_panel_solde_years(
    years: List[int],
    input_pattern: str,
    output_key: str | None,
    bucket: str,
    s3_client,
):
    dfs = []
    for y in years:
        key = input_pattern.format(year=y)
        try:
            df = read_parquet_s3(key, bucket=bucket, s3_client=s3_client)
            df.columns = df.columns.str.strip().str.upper()
            dfs.append(df)
            print(f"✔️ Chargé : {key}  ({df.shape[0]} lignes, {df.shape[1]} colonnes)")
        except ClientError as e:
            code = e.response.get("Error", {}).get("Code", "")
            if code in ("NoSuchKey", "404", "NotFound"):
                print(f"⚠️ Manquant (ignoré) : {key}")
            else:
                print(f"⚠️ Erreur sur {key} (ignoré) : {e}")
        except Exception as e:
            print(f"⚠️ Erreur sur {key} (ignoré) : {e}")

    if not dfs:
        raise RuntimeError("Aucune table chargée. Vérifie les années/chemins.")

    df_final = pd.concat(dfs, axis=0, join="outer", ignore_index=True)
    print("Colonnes finales :", df_final.columns.tolist())
    print(f"Shape final : {df_final.shape[0]} lignes x {df_final.shape[1]} colonnes")

    if output_key is None:
        output_key = f"Solde/Panel_solde_1_2_{min(years)}_{max(years)}.parquet"

    save_parquet_to_s3(df_final, object_key=output_key, bucket=bucket, s3_client=s3_client)
    gc.collect()
    return df_final



## 3) DÉFINITION DES PARAMÈTRES

In [2]:
ENDPOINT_URL = "http://minio.mon-namespace.svc.cluster.local:80"
AWS_KEY      = "wer1Or8j7hXa3QGk2M3M"
AWS_SECRET   = "YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt"
VERIFY_SSL   = False

BUCKET       = "admindataanstat"
KEY_PANEL_2  = "Solde/panel_solde_df_2.parquet"
KEY_PANEL_1  = "Solde/panel_solde_df_1_code_QUALITE.parquet"

# Colonnes à conserver (None = toutes)
COLUMNS_PANEL_2_MIN = None
COLUMNS_PANEL_1_MIN = None


## 4) PIPELINE PRINCIPAL - VERSION OPTIMISÉE

In [3]:
# Configuration fusion
DISABLE_XLSX    = True
MAX_XLSX_ROWS   = 200_000
XLSX_CHUNK_ROWS = 1_000_000
XLSX_ENGINE     = "xlsxwriter"
KEEP_ALL_COLUMNS = True

KEEP_LEFT_COLS  = ["SOURCE_FICHIER", "MONTANT BRUT"]
KEEP_RIGHT_COLS = ["DATE_COLLECTE", "ENTITE", "GRADE"]
EXCLUDE_LEFT_COLS  = []
EXCLUDE_RIGHT_COLS = []

# Années à traiter
YEARS_TO_PROCESS = [2021, 2022, 2023, 2024, 2025]


### 4.1 Fonction optimisée : traitement année par année

In [4]:

def process_one_year_optimized(y: int, s3_client):
    """
    Charge, consolide et fusionne les données pour une seule année.
    Libère la mémoire après chaque étape.
    """
    print(f"\n{'='*70}")
    print(f"  TRAITEMENT ANNÉE {y}")
    print(f"{'='*70}\n")
    
    # ---- ÉTAPE 1: Charger Panel 2 (montants) ----
    print(f"📥 Chargement Panel 2 pour {y}...")
    panel_2_full = read_parquet_from_s3(
        BUCKET, KEY_PANEL_2,
        endpoint_url=ENDPOINT_URL,
        aws_access_key_id=AWS_KEY,
        aws_secret_access_key=AWS_SECRET,
        verify=VERIFY_SSL,
        columns=COLUMNS_PANEL_2_MIN
    )
    
    # Filtrer immédiatement pour l'année
    ymL = _year_month_from_source_file(panel_2_full["SOURCE_FICHIER"])
    panel_2_full["YEAR"] = ymL["YEAR"]
    panel_2_year = panel_2_full[panel_2_full["YEAR"] == y].copy()
    del panel_2_full, ymL
    gc.collect()
    
    print(f"   ✓ Panel 2 ({y}): {len(panel_2_year):,} lignes, {panel_2_year.shape[1]} colonnes")
    
    # ---- ÉTAPE 2: Consolider les colonnes répétées ----
    print(f"🔧 Consolidation des colonnes répétées...")
    panel_2_year = consolidate_all_years_and_drop_sources(
        panel_2_year,
        source_col="SOURCE_FICHIER",
        mode="first_non_null",
        treat_zero_as_filled=True,
    )
    print(f"   ✓ Après consolidation: {panel_2_year.shape[1]} colonnes")
    gc.collect()
    
    # ---- ÉTAPE 3: Charger Panel 1 (infos admin) ----
    print(f"📥 Chargement Panel 1 pour {y}...")
    panel_1_full = read_parquet_from_s3(
        BUCKET, KEY_PANEL_1,
        endpoint_url=ENDPOINT_URL,
        aws_access_key_id=AWS_KEY,
        aws_secret_access_key=AWS_SECRET,
        verify=VERIFY_SSL,
        columns=COLUMNS_PANEL_1_MIN
    )
    
    # Filtrer immédiatement pour l'année
    ymR = _year_month_from_date_collecte(panel_1_full["DATE_COLLECTE"])
    panel_1_full["YEAR"] = ymR["YEAR"]
    panel_1_year = panel_1_full[panel_1_full["YEAR"] == y].copy()
    del panel_1_full, ymR
    gc.collect()
    
    print(f"   ✓ Panel 1 ({y}): {len(panel_1_year):,} lignes, {panel_1_year.shape[1]} colonnes")
    
    # ---- ÉTAPE 4: Préparer les colonnes à garder ----
    if KEEP_ALL_COLUMNS:
        keep_left  = None
        keep_right = None
        
        if EXCLUDE_LEFT_COLS or EXCLUDE_RIGHT_COLS:
            cols_left  = [c for c in panel_2_year.columns if c not in EXCLUDE_LEFT_COLS]
            cols_right = [c for c in panel_1_year.columns if c not in EXCLUDE_RIGHT_COLS]
            keep_left, keep_right = cols_left, cols_right
    else:
        keep_left, keep_right = KEEP_LEFT_COLS, KEEP_RIGHT_COLS
    
    # ---- ÉTAPE 5: Fusion ----
    print(f"🔗 Fusion des deux panels...")
    merged = merge_union_for_year_with_suffix(
        panel_2_year, panel_1_year,
        year=y,
        join_on_month=True,
        suffix_left="_b0",
        suffix_right="_b1",
        dedup_right=True,
        export_unmatched_prefix=None if DISABLE_XLSX else "bilan_merge",
        xlsx_chunk_rows=XLSX_CHUNK_ROWS,
        xlsx_engine=XLSX_ENGINE,
        keep_left=keep_left,
        keep_right=keep_right,
        max_xlsx_rows=MAX_XLSX_ROWS
    )
    
    print(f"   ✓ Colonnes fusionnées: {merged.shape[1]}")
    
    # ---- ÉTAPE 6: Sauvegarde ----
    print(f"💾 Sauvegarde du résultat...")
    save_panel_solde_year_parquet(
        merged, 
        year=y, 
        prefix="Solde/Panel_solde_1_2_",
        bucket=BUCKET,
        s3_client=s3_client
    )
    
    # ---- ÉTAPE 7: Nettoyage mémoire ----
    del panel_2_year, panel_1_year, merged
    gc.collect()
    
    print(f"✅ Année {y} terminée avec succès!\n")




  TRAITEMENT ANNÉE 2015

📥 Chargement Panel 2 pour 2015...
   ✓ Panel 2 (2015): 2,278,823 lignes, 191 colonnes
🔧 Consolidation des colonnes répétées...
   ✓ Après consolidation: 184 colonnes
📥 Chargement Panel 1 pour 2015...
   ✓ Panel 1 (2015): 2,304,670 lignes, 83 colonnes
🔗 Fusion des deux panels...
Fusion OUTER 2015 (join_on_month=True) → 2,305,311 lignes, 271 colonnes

Bilan _merge :
_merge
both          2278182
right_only      26488
left_only         641
Name: count, dtype: int64
   ✓ Colonnes fusionnées: 271
💾 Sauvegarde du résultat...
✅ Sauvegardé : s3://admindataanstat/Solde/Panel_solde_1_2_2015.parquet
✅ Année 2015 terminée avec succès!


  TRAITEMENT ANNÉE 2016

📥 Chargement Panel 2 pour 2016...
   ✓ Panel 2 (2016): 2,196,211 lignes, 191 colonnes
🔧 Consolidation des colonnes répétées...
   ✓ Après consolidation: 184 colonnes
📥 Chargement Panel 1 pour 2016...
   ✓ Panel 1 (2016): 2,427,817 lignes, 83 colonnes
🔗 Fusion des deux panels...
Fusion OUTER 2016 (join_on_month=True)

/tmp/ipykernel_123/1918366011.py:618: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(dfs, axis=0, join="outer", ignore_index=True)
/tmp/ipykernel_123/1918366011.py:618: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_final = pd.concat(dfs, axis=0, join="outer", ignore_index=True)
/tmp/ipykernel_123/1918366011.py:618: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all

Colonnes finales : ['SOLDE DE BASE', 'SALAIRE D AGENT TEMPORAIRE', 'SALAIRE DE GENS DE MAISON', 'FORFAIT INDICIAIRE AMBASSADEUR', 'COMPENSATRICE SMIG', 'REVALORISATION 2014', 'COMPENSATRICE NON IMP AMBASSADES', 'RISQUE SANITAIRE', 'COMPENSATRICE NON IMPOSABLE', 'COMPENSATRICE IMPOSABLE', 'COMPLEMENT INP HB', 'SUJETION GREFFIER EN CHEF', 'PRIME DE RECHERCHE', 'COMPENSATRICE AMBASSADEUR', 'CONTRAINTE', 'HEURES SUPPL , PERSONNEL ENSEIGNANT', 'HEURES DE COURS', 'PATROUILLE', 'TRAITEMENT PERSONNEL NON CLASSE', 'HEURES SUPPLEMENTAIRES ADMINISTRATION', 'FORFAIT CAP', 'TRAITEMENT CHEF DE CANTON', 'UTILISATION VEHICULE P/PPNAT', 'RISQUE', 'VERIFICATION CHAMBRE CPTES/POINCON OR', 'TRANSPORT', 'PRIME DE CAISSE', 'ENTRETIEN UNIFORME', 'ACQUISITION UNIFORME', 'DEPLACEMENT TEMPORAIRE', 'ALLOCATION ELEVE', 'TRAITEMENT AMBASSADEUR', 'INDEMNITE INGENIEUR SYSTEME', 'REGULARISATION CN EXERCICE ANTERIEUR', 'REGULARISATION IGR EXERCICE ANTERIEUR', 'REGULARISATION IGR', 'REGULARISATION CN', 'REGULARISATION 

### 4.2 Exécution principale

In [ ]:

if __name__ == "__main__":
    # Créer le client S3
    s3_client = get_s3_client(ENDPOINT_URL, AWS_KEY, AWS_SECRET, VERIFY_SSL)
    
    # Traiter chaque année séparément
    for year in YEARS_TO_PROCESS:
        try:
            process_one_year_optimized(year, s3_client)
        except Exception as e:
            print(f"❌ ERREUR lors du traitement de l'année {year}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    print("\n" + "="*70)
    print("  TOUTES LES ANNÉES ONT ÉTÉ TRAITÉES")
    print("="*70 + "\n")
    
    # ---- ÉTAPE FINALE: Empilement ----
    print("📚 Empilement de toutes les années...")
    try:
        df_empile = stack_panel_solde_years(
            years=YEARS_TO_PROCESS,
            input_pattern="Solde/Panel_solde_1_2_{year}.parquet",
            output_key=None,
            bucket=BUCKET,
            s3_client=s3_client,
        )
        print(f"\n✅ EMPILEMENT TERMINÉ: {df_empile.shape}")
    except Exception as e:
        print(f"❌ ERREUR lors de l'empilement: {e}")
        import traceback
        traceback.print_exc()

# SI CETTE ETAPE EST FRANCHIE RESTE A FUSIONNER Panel_solde_1_2_2015_2020.parquet à Panel_solde_1_2_2021_2025.parquet POUR OBTENIR UNE BASE QUI PART DE 2015 A 2021